# 02 — LLM-as-Judge Baseline (Claude Sonnet 4.6 via Claude Code)

**Goal:** Establish the Claude Sonnet 4.6 macro-F1 baseline on Banking77. The number this notebook produces is the horizontal line we're trying to match with the fine-tuned DistilBERT in notebook 03.

## Why Claude Code instead of the Anthropic API

Claude Code (Sonnet 4.6) and the Anthropic API serve the same model weights. Claude Code runs against my existing Claude Max subscription so the project costs $0; the API would cost ~$25 across iterations. Tradeoff: I lose per-call latency measurements and have to paste prompts manually instead of looping. For anyone reproducing this study with API access, the same prompts work — see `src/prompts/v*.txt`.

## Iteration strategy

Running each prompt version on the full 3,080-row test set is expensive in paste-time (~30-40 min). So I iterate on the 300-row dev slice (carved from train pool in notebook 01, disjoint from test) until the prompt looks solid, then run the chosen version once on the full test set.

## Sections

1. Setup — load data, prompt template, pick version + dataset
2. Generate batch files (paste-ready)
3. Status check — which batches have responses?
4. Parse responses → predictions dataframe
5. Score — accuracy, macro-F1, top confusion pairs, failure-mode hints for the next prompt version
6. Save predictions + per-iteration summary
7. Final test eval (only run with v_final after dev iteration converges)

## 1. Setup

**Change these constants** when iterating to a new prompt version or moving from dev to final test eval.

In [1]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.data import load_test_set, load_dev_slice, load_label_names
from src.prompts import load_prompt, list_versions
from src.llm_eval import build_batch_prompt, parse_response
from src.eval import compute_metrics

# === iteration knobs ===
PROMPT_VERSION = 'v1'        # change this when iterating: v1 → v2 → v_final
DATASET = 'dev_slice'        # 'dev_slice' while iterating; 'test' for the final eval
BATCH_SIZE = 150             # queries per paste; tune if Claude's output gets truncated

# === load data ===
dev_df = load_dev_slice()
test_df = load_test_set()
label_names = load_label_names()

target_df = {'dev_slice': dev_df, 'test': test_df}[DATASET].reset_index(drop=True)

# === paths ===
BATCH_DIR = Path('../results/eval_batches') / PROMPT_VERSION / DATASET
RESPONSE_DIR = Path('../results/eval_responses') / PROMPT_VERSION / DATASET
BATCH_DIR.mkdir(parents=True, exist_ok=True)
RESPONSE_DIR.mkdir(parents=True, exist_ok=True)

n_batches = math.ceil(len(target_df) / BATCH_SIZE)

print(f'Available prompt versions: {list_versions()}')
print(f'Selected:                  {PROMPT_VERSION}')
print(f'Dataset:                   {DATASET} ({len(target_df):,} rows)')
print(f'Batches:                   {n_batches} of up to {BATCH_SIZE}')
print(f'Batch dir:                 {BATCH_DIR}')
print(f'Response dir:              {RESPONSE_DIR}')

Available prompt versions: ['v1']
Selected:                  v1
Dataset:                   dev_slice (300 rows)
Batches:                   2 of up to 150
Batch dir:                 ../results/eval_batches/v1/dev_slice
Response dir:              ../results/eval_responses/v1/dev_slice


## 2. Generate batch files

Each batch file is a paste-ready prompt for one Claude Code session. Open the file, copy the full contents (`cat <path> | pbcopy` on Mac), paste into Claude Code with Sonnet 4.6 selected, save the response back to `RESPONSE_DIR/batch_NN.txt`.

In [2]:
template = load_prompt(PROMPT_VERSION)

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(target_df))
    batch_df = target_df.iloc[start:end]

    prompt_text = build_batch_prompt(
        template=template,
        queries=batch_df['text'].tolist(),
        label_names=label_names,
        start_id=start + 1,
    )

    batch_path = BATCH_DIR / f'batch_{batch_idx + 1:02d}.txt'
    batch_path.write_text(prompt_text)

    n_chars = len(prompt_text)
    print(f'  batch_{batch_idx+1:02d}.txt  ({end - start} rows, ~{n_chars:,} chars, ~{n_chars // 4:,} tokens)')

print()
print('=== Paste-and-parse workflow ===')
print(f'For each batch:')
print(f'  1. Open the batch file, copy contents (e.g. `cat {BATCH_DIR}/batch_01.txt | pbcopy`)')
print(f'  2. Open a fresh terminal in any directory OTHER than this repo')
print(f'  3. Run: claude')
print(f'  4. /model claude-sonnet-4-6   (verify Sonnet 4.6 in the status bar)')
print(f'  5. Paste, hit enter, wait for the JSON array response')
print(f'  6. Copy the JSON array; save to: {RESPONSE_DIR}/batch_NN.txt')
print(f'  7. Re-run sections 3-6 of this notebook to parse + score')

  batch_01.txt  (150 rows, ~12,062 chars, ~3,015 tokens)
  batch_02.txt  (150 rows, ~12,206 chars, ~3,051 tokens)

=== Paste-and-parse workflow ===
For each batch:
  1. Open the batch file, copy contents (e.g. `cat ../results/eval_batches/v1/dev_slice/batch_01.txt | pbcopy`)
  2. Open a fresh terminal in any directory OTHER than this repo
  3. Run: claude
  4. /model claude-sonnet-4-6   (verify Sonnet 4.6 in the status bar)
  5. Paste, hit enter, wait for the JSON array response
  6. Copy the JSON array; save to: ../results/eval_responses/v1/dev_slice/batch_NN.txt
  7. Re-run sections 3-6 of this notebook to parse + score


## 3. Status check

Which batches have responses saved? Run this between paste sessions to track progress.

In [3]:
status_rows = []
for batch_idx in range(n_batches):
    response_path = RESPONSE_DIR / f'batch_{batch_idx + 1:02d}.txt'
    status = 'ready' if response_path.exists() else 'MISSING — paste it'
    size_kb = response_path.stat().st_size / 1024 if response_path.exists() else 0
    status_rows.append({'batch': batch_idx + 1, 'status': status, 'response_kb': round(size_kb, 1)})

status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))

n_ready = (status_df['status'] == 'ready').sum()
print(f'\n{n_ready}/{n_batches} batches have responses. ', end='')
if n_ready < n_batches:
    print(f'Paste the remaining {n_batches - n_ready} before running section 4.')
else:
    print('All ready — proceed to section 4.')

 batch             status  response_kb
     1 MISSING — paste it            0
     2 MISSING — paste it            0

0/2 batches have responses. Paste the remaining 2 before running section 4.


## 4. Parse responses → predictions dataframe

Reads every saved response, parses (tolerant of markdown wrappers), validates intent names, joins against ground truth from the target dataset.

In [4]:
all_predictions = []
parse_issues = []
n_responses_found = 0

for batch_idx in range(n_batches):
    response_path = RESPONSE_DIR / f'batch_{batch_idx + 1:02d}.txt'
    if not response_path.exists():
        continue
    n_responses_found += 1

    response_text = response_path.read_text()
    parsed, warning = parse_response(response_text)

    if parsed is None:
        parse_issues.append(f'batch {batch_idx + 1}: parse failed — {warning}')
        continue
    if warning:
        parse_issues.append(f'batch {batch_idx + 1}: {warning}')

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(target_df))
    if len(parsed) != end - start:
        parse_issues.append(f'batch {batch_idx + 1}: expected {end - start} predictions, got {len(parsed)}')

    for item in parsed:
        if not isinstance(item, dict):
            parse_issues.append(f'batch {batch_idx + 1}: item is not a dict: {item!r}')
            continue
        all_predictions.append({
            'batch': batch_idx + 1,
            'pred_id': item.get('id'),
            'pred_intent': str(item.get('intent', '')).strip(),
        })

if n_responses_found == 0:
    print(f'No responses found in {RESPONSE_DIR}/ yet.')
    print('Paste batches into Claude Code, save responses, then re-run this section.')
    pred_df = pd.DataFrame()
else:
    pred_df = pd.DataFrame(all_predictions)

    truth_df = target_df.copy()
    truth_df['pred_id'] = truth_df.index + 1
    truth_df['true_intent'] = truth_df['label'].map(lambda i: label_names[int(i)])

    pred_df = pred_df.merge(truth_df[['pred_id', 'text', 'true_intent']], on='pred_id', how='left')

    labels_set = set(label_names)
    pred_df['is_valid_intent'] = pred_df['pred_intent'].isin(labels_set)
    pred_df['is_correct'] = pred_df['pred_intent'] == pred_df['true_intent']

    print(f'Responses found:   {n_responses_found} / {n_batches}')
    print(f'Total predictions: {len(pred_df):,} (target rows: {len(target_df):,})')
    print(f'Hallucinated:      {(~pred_df["is_valid_intent"]).sum()}')
    print(f'Correct:           {pred_df["is_correct"].sum():,} / {len(pred_df):,}  ({100 * pred_df["is_correct"].mean():.2f}%)')

    if parse_issues:
        print(f'\nParse issues ({len(parse_issues)}):')
        for x in parse_issues[:10]:
            print(f'  - {x}')
        if len(parse_issues) > 10:
            print(f'  ... and {len(parse_issues) - 10} more')

No responses found in ../results/eval_responses/v1/dev_slice/ yet.
Paste batches into Claude Code, save responses, then re-run this section.


## 5. Score + failure-mode analysis

Headline metrics + the most-confused intent pairs. The confusion pairs are where you'll see Banking77's label noise (e.g. `get_physical_card` vs `order_physical_card`) AND where the prompt's weaknesses show up — feed these patterns into the next prompt version.

In [5]:
if pred_df.empty:
    print('Nothing to score yet — run section 4 once responses are saved.')
else:
    y_true = pred_df['true_intent'].values
    y_pred_for_metric = pred_df['pred_intent'].where(pred_df['is_valid_intent'], '__hallucinated__').values

    metrics = compute_metrics(y_true, y_pred_for_metric, labels=label_names)

    print(f'=== {PROMPT_VERSION} on {DATASET} ({len(pred_df):,} predictions) ===')
    print(f'Accuracy:    {metrics["accuracy"]:.4f}')
    print(f'Macro F1:    {metrics["macro_f1"]:.4f}')
    print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')

    wrong = pred_df[(~pred_df['is_correct']) & pred_df['is_valid_intent']]
    confusion_counts = (
        wrong.groupby(['true_intent', 'pred_intent']).size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
    print(f'\nTop 20 confusion pairs:')
    print(confusion_counts.head(20).to_string(index=False) if not confusion_counts.empty else '  (none — all valid predictions correct)')

Nothing to score yet — run section 4 once responses are saved.


In [6]:
if pred_df.empty:
    print('Nothing to inspect yet — run section 4 once responses are saved.')
else:
    print('Sample of wrong predictions (first 15) — feed these patterns into the next prompt version:')
    for _, row in wrong.head(15).iterrows():
        print(f'  TRUE:  {row["true_intent"]}')
        print(f'  PRED:  {row["pred_intent"]}')
        print(f'  QUERY: {row["text"]}')
        print()

Nothing to inspect yet — run section 4 once responses are saved.


## 6. Save predictions + summary

Writes per-prediction parquet (joined to notebook 04 for the bootstrap test) and a small summary JSON so you can compare prompt versions at a glance.

In [7]:
if pred_df.empty:
    print('Nothing to save yet — run section 4 once responses are saved.')
else:
    preds_path = Path(f'../results/llm_predictions_{PROMPT_VERSION}_{DATASET}.parquet')
    pred_df.to_parquet(preds_path)
    print(f'Saved {preds_path}')

    summary = {
        'prompt_version': PROMPT_VERSION,
        'dataset': DATASET,
        'n_predictions': int(len(pred_df)),
        'n_hallucinated': int((~pred_df['is_valid_intent']).sum()),
        'accuracy': float(metrics['accuracy']),
        'macro_f1': float(metrics['macro_f1']),
        'weighted_f1': float(metrics['weighted_f1']),
        'top_confusions': confusion_counts.head(10).to_dict('records'),
    }
    summary_path = Path(f'../results/llm_summary_{PROMPT_VERSION}_{DATASET}.json')
    summary_path.write_text(json.dumps(summary, indent=2))
    print(f'Saved {summary_path}')

Nothing to save yet — run section 4 once responses are saved.


## 7. Final test eval (run only with v_final)

Once dev iteration converges and you've picked v_final, set `PROMPT_VERSION = 'v_final'` and `DATASET = 'test'` at the top, then re-run sections 1-6. The predictions saved to `results/llm_predictions_v_final_test.parquet` become the LLM baseline that notebook 04 compares DistilBERT against.

Expected scale: ~21 batches of 150 rows = ~30-40 minutes of focused paste-and-parse work.